# K-MIMIC-MEDS — ETL Pipeline Validation

This notebook demonstrates and validates the ETL pipeline that converts the **Synthetic K-MIMIC (SYN-ICU)** Korean ICU dataset into the **MEDS (Medical Event Data Standard)** format.

---

## What is MEDS?

MEDS is an international standard for longitudinal medical event data, designed for reproducible machine learning research in healthcare. Every medical observation — a lab result, a vital sign, a diagnosis, a procedure — is represented as a single row with 4 columns:

| Column | Type | Description |
|---|---|---|
| `subject_id` | int64 | Unique patient identifier |
| `time` | timestamp | When the event occurred (`null` = static, e.g. gender) |
| `code` | string | What happened, in hierarchical format (e.g. `CHARTEVENT//HR///min`) |
| `numeric_value` | float32 | Measured value if applicable |

---

## Pipeline Overview

```
data/raw/*.xlsx            (15 source tables, Korean ICU data)
        │
        ▼
   pre_meds.py             (clean IDs, fix timestamps, normalize columns)
        │
        ▼
data/intermediate/*.parquet
        │
        ▼
  meds_convert.py          (extract events, build codes, split patients)
        │
        ▼
data/output/               (MEDS-compliant dataset)
├── data/
│   ├── train/0.parquet
│   ├── tuning/0.parquet
│   └── held_out/0.parquet
└── metadata/
    ├── codes.parquet
    ├── dataset.json
    └── subject_splits.parquet
```

---

## Key Engineering Decisions

| Challenge | Solution |
|---|---|
| K-MIMIC uses UUID string IDs | SHA-256 hashing → stable int64 |
| De-identified years exceed 2262 (pandas ns limit) | Use `datetime64[us]` (microsecond precision) |
| Korean medical units (`회/min`, `℃`, `×10³/㎕`) | `UNIT_MAP` normalization to international standards |
| Diagnoses ICD had no timestamps | Joined with admissions table on `hadm_id` |
| MEDS-Extract CLI incompatible with Windows | Standalone Python pipeline, same output |
| `iterrows()` too slow on 400k+ rows | Vectorized pandas operations (10-50x faster) |

In [1]:
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("data/output")

# Clinical plausibility ranges for distributional checks
CLINICAL_RANGES = {
    "CHARTEVENT//001C_1021": (0,   300),   # Heart rate (bpm)
    "CHARTEVENT//001C_1023": (0,   60),    # Respiratory rate (/min)
    "CHARTEVENT//001C_1026": (30,  45),    # Body temperature (Celsius)
    "CHARTEVENT//001C_1012": (40,  250),   # Systolic BP (mmHg)
    "CHARTEVENT//001C_1013": (20,  180),   # Diastolic BP (mmHg)
    "CHARTEVENT//001C_1003": (50,  100),   # SpO2 (%)
    "LAB//001L2001":         (0,   100),   # WBC (x10e3/uL)
    "LAB//001L2003":         (3,   20),    # Hemoglobin (g/dL)
    "LAB//001L3005":         (40,  500),   # Glucose (mg/dL)
}

print("Libraries loaded.")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Libraries loaded.
Output directory: C:\Users\33633\Desktop\Efrei\M2\Stage\K-MIMICS-MEDS\data\output


---
## 1. Dataset Metadata

In [3]:
meta = json.loads((OUTPUT_DIR / "metadata/dataset.json").read_text())

print("Dataset metadata:")
print("-" * 40)
for key, value in meta.items():
    print(f"  {key:<20} : {value}")

Dataset metadata:
----------------------------------------
  dataset_name         : SYN-ICU
  dataset_version      : 1.0
  etl_name             : kmimic-meds
  etl_version          : 0.2.0
  meds_version         : 0.3.3
  created_at           : 2026-04-16T04:38:40.719663+00:00


---
## 2. Output File Structure

In [4]:
expected = [
    "data/train/0.parquet",
    "data/tuning/0.parquet",
    "data/held_out/0.parquet",
    "metadata/codes.parquet",
    "metadata/dataset.json",
    "metadata/subject_splits.parquet",
]

print("File structure check:")
print("-" * 50)
all_present = True
for f in expected:
    path = OUTPUT_DIR / f
    exists = path.exists()
    size = f"{path.stat().st_size / 1024:.1f} KB" if exists else ""
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {f:<42} {size}")
    if not exists:
        all_present = False

print()
print("✅ All files present." if all_present else "❌ WARNING: some files are missing.")

File structure check:
--------------------------------------------------
  ✅  data/train/0.parquet                       4577.8 KB
  ✅  data/tuning/0.parquet                      630.3 KB
  ✅  data/held_out/0.parquet                    646.3 KB
  ✅  metadata/codes.parquet                     7.2 KB
  ✅  metadata/dataset.json                      0.2 KB
  ✅  metadata/subject_splits.parquet            14.2 KB

✅ All files present.


---
## 3. MEDS Schema Compliance

In [5]:
train = pd.read_parquet(OUTPUT_DIR / "data/train/0.parquet")

print("Column types:")
print("-" * 40)
print(train.dtypes)
print()

schema_checks = {
    "subject_id is int64"      : str(train["subject_id"].dtype) in ("int64", "Int64"),
    "time is datetime[us]"     : "datetime" in str(train["time"].dtype),
    "code is string"           : str(train["code"].dtype) in ("object", "string", "str"),
    "numeric_value is float32" : str(train["numeric_value"].dtype) == "float32",
    "exactly 4 columns"        : len(train.columns) == 4,
}

print("Schema checks:")
print("-" * 40)
for check, result in schema_checks.items():
    print(f"  {'✅' if result else '❌'}  {check}")

Column types:
----------------------------------------
subject_id                Int64
time             datetime64[us]
code                        str
numeric_value           float32
dtype: object

Schema checks:
----------------------------------------
  ✅  subject_id is int64
  ✅  time is datetime[us]
  ✅  code is string
  ✅  numeric_value is float32
  ✅  exactly 4 columns


---
## 4. Patient Record Preview

Complete chronological record of one patient. Static events (no timestamp) come first, then dynamic events in chronological order.

In [6]:
first_patient = train["subject_id"].iloc[0]
patient_df = train[train["subject_id"] == first_patient].reset_index(drop=True)

print(f"Patient {first_patient} — {len(patient_df)} total events")
print()
patient_df.head(20)

Patient 4056806253170987 — 681 total events



,subject_id,time,code,numeric_value
0,4056806253170987,NaT,GENDER//M,NaN
1,4056806253170987,2090-01-01 00:00:01,MEDS_BIRTH,NaN
2,4056806253170987,2130-10-29 11:32:28,ED_REGISTRATION,NaN
3,4056806253170987,2130-10-29 11:53:28,ED_OUT,NaN
4,4056806253170987,2130-10-29 12:38:28,HOSPITAL_ADMISSION//Emergency department,NaN
5,4056806253170987,2130-10-29 12:38:28,INSURANCE//Others,NaN
6,4056806253170987,2130-10-29 12:38:28,MARITAL_STATUS//single,NaN
7,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//I251,NaN
8,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//I839,NaN
9,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//Others,NaN


In [7]:
print("Event type breakdown for this patient:")
print("-" * 40)
prefixes = patient_df["code"].str.split("//").str[0].value_counts()
for prefix, count in prefixes.items():
    print(f"  {prefix:<25} : {count} events")

Event type breakdown for this patient:
----------------------------------------
  PROCEDURE_START           : 209 events
  PROCEDURE_END             : 209 events
  CHARTEVENT                : 178 events
  LAB                       : 42 events
  OUTPUT                    : 16 events
  INPUT_START               : 12 events
  DIAGNOSIS                 : 3 events
  GENDER                    : 1 events
  MEDS_BIRTH                : 1 events
  ED_REGISTRATION           : 1 events
  ED_OUT                    : 1 events
  HOSPITAL_ADMISSION        : 1 events
  INSURANCE                 : 1 events
  MARITAL_STATUS            : 1 events
  ICU_ADMISSION             : 1 events
  ICU_DISCHARGE             : 1 events
  PROCEDURE_ICD             : 1 events
  HOSPITAL_DISCHARGE        : 1 events
  MEDICATION                : 1 events


---
## 5. Dataset Statistics

In [8]:
all_dfs = []
for split_name in ["train", "tuning", "held_out"]:
    df = pd.read_parquet(OUTPUT_DIR / f"data/{split_name}/0.parquet")
    df["split"] = split_name
    all_dfs.append(df)

full = pd.concat(all_dfs, ignore_index=True)

print("=" * 45)
print("GLOBAL STATISTICS")
print("=" * 45)
print(f"  Total events          : {len(full):>12,}")
print(f"  Total patients        : {full['subject_id'].nunique():>12,}")
print(f"  Static events (NaT)   : {full['time'].isna().sum():>12,}")
print(f"  Dynamic events        : {full['time'].notna().sum():>12,}")
print(f"  With numeric value    : {full['numeric_value'].notna().sum():>12,}")
print(f"  Unique codes          : {full['code'].nunique():>12,}")
print()
print("Events and patients per split:")
print("-" * 45)
summary = full.groupby("split").agg(
    events=("code", "count"),
    patients=("subject_id", "nunique")
)
print(summary)

GLOBAL STATISTICS
  Total events          :    1,381,580
  Total patients        :        1,328
  Static events (NaT)   :        1,328
  Dynamic events        :    1,380,252
  With numeric value    :      605,941
  Unique codes          :          204

Events and patients per split:
---------------------------------------------
           events  patients
split                      
held_out   138886       134
train     1106508      1062
tuning     136186       132


---
## 6. Event Type Distribution

In [9]:
full["event_type"] = full["code"].str.split("//").str[0]

print("Event distribution by type:")
print("-" * 55)
dist = full["event_type"].value_counts()
for event_type, count in dist.items():
    pct = count / len(full) * 100
    bar = "█" * int(pct / 2)
    print(f"  {event_type:<25} : {count:>9,}  ({pct:4.1f}%)  {bar}")

Event distribution by type:
-------------------------------------------------------
  CHARTEVENT                :   416,645  (30.2%)  ███████████████
  PROCEDURE_START           :   371,107  (26.9%)  █████████████
  PROCEDURE_END             :   371,107  (26.9%)  █████████████
  LAB                       :   137,662  (10.0%)  ████
  OUTPUT                    :    36,353  ( 2.6%)  █
  INPUT_START               :    26,691  ( 1.9%)  
  MEDICATION                :     5,675  ( 0.4%)  
  DIAGNOSIS                 :     3,091  ( 0.2%)  
  PROCEDURE_ICD             :     1,479  ( 0.1%)  
  GENDER                    :     1,328  ( 0.1%)  
  HOSPITAL_ADMISSION        :     1,328  ( 0.1%)  
  MARITAL_STATUS            :     1,328  ( 0.1%)  
  ICU_ADMISSION             :     1,328  ( 0.1%)  
  ICU_DISCHARGE             :     1,328  ( 0.1%)  
  HOSPITAL_DISCHARGE        :     1,328  ( 0.1%)  
  INSURANCE                 :     1,327  ( 0.1%)  
  MEDS_BIRTH                :     1,249  ( 0.1%)  
  E

---
## 7. Patient Split Distribution

In [10]:
splits = pd.read_parquet(OUTPUT_DIR / "metadata/subject_splits.parquet")
total = len(splits)
counts = splits["split"].value_counts()

print("Patient split distribution:")
print("-" * 40)
for split_name in ["train", "tuning", "held_out"]:
    count = counts.get(split_name, 0)
    pct = count / total * 100
    print(f"  {split_name:<12} : {count:>5} patients ({pct:.1f}%)")

print()
split_checks = {
    "train ~80%"    : 75 <= counts.get("train", 0) / total * 100 <= 85,
    "tuning ~10%"   : 8  <= counts.get("tuning", 0) / total * 100 <= 12,
    "held_out ~10%" : 8  <= counts.get("held_out", 0) / total * 100 <= 12,
}
for check, result in split_checks.items():
    print(f"  {'✅' if result else '❌'}  {check}")

Patient split distribution:
----------------------------------------
  train        :  1062 patients (80.0%)
  tuning       :   132 patients (9.9%)
  held_out     :   134 patients (10.1%)

  ✅  train ~80%
  ✅  tuning ~10%
  ✅  held_out ~10%


---
## 8. Code Catalog

In [11]:
codes = pd.read_parquet(OUTPUT_DIR / "metadata/codes.parquet")

print(f"Total unique codes          : {len(codes)}")
print(f"With description            : {codes['description'].notna().sum()}")
print(f"With parent_codes (EDI)     : {codes['parent_codes'].notna().sum()}")
print()
print("Sample — LAB codes with EDI parent codes:")
codes[codes["parent_codes"].notna()][["code", "description", "parent_codes"]].head(8)

Total unique codes          : 204
With description            : 113
With parent_codes (EDI)     : 32

Sample — LAB codes with EDI parent codes:


,code,description,parent_codes
80,LAB//001L2001//x10e3/uL,WBC(검사24시간가능),[EDI/D0002014]
81,LAB//001L2002//x10e6/uL,RBC(검사24시간가능),[EDI/D0002034]
82,LAB//001L2003//g/dL,Hb(검사24시간가능),[EDI/D0002054]
83,LAB//001L2004//%,Hct,[EDI/D0002044]
84,LAB//001L2005//%,RDW(검사24시간가능),[EDI/D0002024]
85,LAB//001L2006//fL,PDW(검사24시간가능),[EDI/D0002064]
86,LAB//001L2009//x10e3/uL,PLT(검사24시간가능),[EDI/D0002074]
102,LAB//001L2204//sec,aPTT (검사24시간가능),[EDI/D1004004]


---
## 9. Temporal Consistency

Verify that event chronology is medically plausible for each patient.

> **Known limitation:** `dod` in K-MIMIC is a date-only field (no time component), stored as midnight (`00:00:00`). Clinical measurements recorded on the same day as death but with a precise timestamp (e.g. `00:45:32`) can appear after `MEDS_DEATH`. A **48-hour tolerance** is applied to account for this. Additionally, `PROCEDURE_START`/`PROCEDURE_END` events are excluded from the death check as they are a known artifact of the synthetic data generation process.

In [12]:
dynamic = full[full["time"].notna()].copy()
dynamic["time"] = pd.to_datetime(dynamic["time"], errors="coerce")

results = {}

# MEDS_BIRTH is the earliest event per patient
birth = dynamic[dynamic["code"] == "MEDS_BIRTH"][["subject_id", "time"]].rename(columns={"time": "birth_time"})
if not birth.empty:
    earliest = dynamic.groupby("subject_id")["time"].min().reset_index().rename(columns={"time": "earliest_time"})
    merged = birth.merge(earliest, on="subject_id")
    violations = merged[merged["birth_time"] > merged["earliest_time"]]
    results["MEDS_BIRTH is earliest event"] = (len(violations) == 0, f"{len(violations)} violations")

# MEDS_DEATH is the latest event (excl. procedure events, 48h tolerance)
death = dynamic[dynamic["code"] == "MEDS_DEATH"][["subject_id", "time"]].rename(columns={"time": "death_time"})
if not death.empty:
    non_proc = dynamic[~dynamic["code"].str.startswith(("PROCEDURE_START", "PROCEDURE_END"))]
    latest = non_proc.groupby("subject_id")["time"].max().reset_index().rename(columns={"time": "latest_time"})
    merged = death.merge(latest, on="subject_id")
    tolerance = pd.Timedelta(hours=48)
    violations = merged[merged["death_time"] + tolerance < merged["latest_time"]]
    results["MEDS_DEATH is latest event (excl. procedures, 48h tolerance)"] = (len(violations) == 0, f"{len(violations)} violations")

# ICU_ADMISSION before ICU_DISCHARGE
icu_in  = dynamic[dynamic["code"].str.startswith("ICU_ADMISSION")][["subject_id", "time"]].rename(columns={"time": "intime"})
icu_out = dynamic[dynamic["code"].str.startswith("ICU_DISCHARGE")][["subject_id", "time"]].rename(columns={"time": "outtime"})
if not icu_in.empty and not icu_out.empty:
    merged = icu_in.groupby("subject_id")["intime"].min().reset_index().merge(
             icu_out.groupby("subject_id")["outtime"].min().reset_index(), on="subject_id")
    violations = merged[merged["intime"] > merged["outtime"]]
    results["ICU_ADMISSION before ICU_DISCHARGE"] = (len(violations) == 0, f"{len(violations)} violations")

# HOSPITAL_ADMISSION before HOSPITAL_DISCHARGE
hadm_in  = dynamic[dynamic["code"].str.startswith("HOSPITAL_ADMISSION")][["subject_id", "time"]].rename(columns={"time": "admittime"})
hadm_out = dynamic[dynamic["code"].str.startswith("HOSPITAL_DISCHARGE")][["subject_id", "time"]].rename(columns={"time": "dischtime"})
if not hadm_in.empty and not hadm_out.empty:
    merged = hadm_in.groupby("subject_id")["admittime"].min().reset_index().merge(
             hadm_out.groupby("subject_id")["dischtime"].min().reset_index(), on="subject_id")
    violations = merged[merged["admittime"] > merged["dischtime"]]
    results["HOSPITAL_ADMISSION before HOSPITAL_DISCHARGE"] = (len(violations) == 0, f"{len(violations)} violations")

# No duplicate GENDER events per patient
gender_counts = full[full["code"].str.startswith("GENDER")].groupby("subject_id").size()
duplicates = (gender_counts > 1).sum()
results["No duplicate GENDER events per patient"] = (duplicates == 0, f"{duplicates} patients with duplicates")

print("Temporal consistency checks:")
print("-" * 60)
for check, (result, detail) in results.items():
    print(f"  {'✅' if result else '❌'}  {check} — {detail}")

Temporal consistency checks:
------------------------------------------------------------
  ✅  MEDS_BIRTH is earliest event — 0 violations
  ✅  MEDS_DEATH is latest event (excl. procedures, 48h tolerance) — 0 violations
  ✅  ICU_ADMISSION before ICU_DISCHARGE — 0 violations
  ✅  HOSPITAL_ADMISSION before HOSPITAL_DISCHARGE — 0 violations
  ✅  No duplicate GENDER events per patient — 0 patients with duplicates


---
## 10. Distributional Checks

Verify that numeric values are within clinically plausible ranges. ICU patients can have extreme but real measurements — a **5% outlier tolerance** is applied.

In [13]:
numeric = full[full["numeric_value"].notna()].copy()

print("Distributional checks (95% of values must be in range):")
print("-" * 70)

for code_prefix, (low, high) in CLINICAL_RANGES.items():
    subset = numeric[numeric["code"].str.startswith(code_prefix)]
    if subset.empty:
        print(f"  ⚠️   {code_prefix.split('//')[1]} — no data found")
        continue
    out_of_range = subset[(subset["numeric_value"] < low) | (subset["numeric_value"] > high)]
    pct_ok = (1 - len(out_of_range) / len(subset)) * 100
    status = "✅" if pct_ok >= 95 else "❌"
    label = code_prefix.split("//")[1]
    print(f"  {status}  {label:<20} range [{low:>4}, {high:>4}] — {pct_ok:.1f}% in range ({len(out_of_range)} outliers / {len(subset):,} total)")

Distributional checks (95% of values must be in range):
----------------------------------------------------------------------
  ✅  001C_1021            range [   0,  300] — 100.0% in range (0 outliers / 33,662 total)
  ✅  001C_1023            range [   0,   60] — 100.0% in range (0 outliers / 33,668 total)
  ✅  001C_1026            range [  30,   45] — 100.0% in range (0 outliers / 26,467 total)
  ✅  001C_1012            range [  40,  250] — 99.6% in range (134 outliers / 31,502 total)
  ✅  001C_1013            range [  20,  180] — 99.8% in range (73 outliers / 31,618 total)
  ✅  001C_1003            range [  50,  100] — 100.0% in range (0 outliers / 20,077 total)
  ✅  001L2001             range [   0,  100] — 100.0% in range (0 outliers / 2,613 total)
  ✅  001L2003             range [   3,   20] — 100.0% in range (0 outliers / 2,467 total)
  ✅  001L3005             range [  40,  500] — 99.5% in range (25 outliers / 4,752 total)


---
## 11. Cross-Cohort Alignment

Verify that the three splits are statistically comparable and that no patient appears in multiple splits.

In [14]:
split_dfs = {}
for split_name in ["train", "tuning", "held_out"]:
    split_dfs[split_name] = pd.read_parquet(OUTPUT_DIR / f"data/{split_name}/0.parquet")

print("Cross-cohort alignment checks:")
print("-" * 60)

# Same event type prefixes across all splits
def get_prefixes(df):
    return set(df["code"].str.split("//").str[0].unique())

train_p  = get_prefixes(split_dfs["train"])
tuning_p = get_prefixes(split_dfs["tuning"])
held_p   = get_prefixes(split_dfs["held_out"])
all_same = (train_p == tuning_p == held_p)
print(f"  {'✅' if all_same else '❌'}  Same event types across all splits")
if not all_same:
    print(f"    Missing in tuning   : {train_p - tuning_p}")
    print(f"    Missing in held_out : {train_p - held_p}")

# Numeric value means comparable across splits
top_code = full[full["numeric_value"].notna()]["code"].value_counts().index[0]
means = {s: df[df["code"] == top_code]["numeric_value"].mean() for s, df in split_dfs.items() if not df[df["code"] == top_code].empty}
if len(means) == 3:
    overall_mean = sum(means.values()) / 3
    max_dev = max(abs(v - overall_mean) / overall_mean * 100 for v in means.values())
    label = top_code.split("//")[1]
    print(f"  {'✅' if max_dev <= 20 else '❌'}  Numeric means comparable across splits ({label}) — max deviation {max_dev:.1f}%")
    for s, m in means.items():
        print(f"    {s:<12} : mean = {m:.3f}")

# No patient in multiple splits
sets = {s: set(df["subject_id"].unique()) for s, df in split_dfs.items()}
overlap = len(sets["train"] & sets["tuning"]) + len(sets["train"] & sets["held_out"]) + len(sets["tuning"] & sets["held_out"])
print(f"  {'✅' if overlap == 0 else '❌'}  No patient appears in multiple splits — {overlap} overlaps")

# Code coverage per split
all_codes = set(full["code"].unique())
for split_name, df in split_dfs.items():
    coverage = len(set(df["code"].unique())) / len(all_codes) * 100
    print(f"  {'✅' if coverage >= 80 else '❌'}  {split_name:<12} covers {coverage:.1f}% of all codes ({len(set(df['code'].unique()))}/{len(all_codes)})")

Cross-cohort alignment checks:
------------------------------------------------------------
  ✅  Same event types across all splits
  ✅  Numeric means comparable across splits (001C_1023_27310) — max deviation 1.7%
    train        : mean = 17.809
    tuning       : mean = 17.882
    held_out     : mean = 18.301
  ✅  No patient appears in multiple splits — 0 overlaps
  ✅  train        covers 100.0% of all codes (204/204)
  ✅  tuning       covers 97.1% of all codes (198/204)
  ✅  held_out     covers 95.1% of all codes (194/204)


---
## 12. Diagnosis Timestamps

Diagnoses are linked to their admission timestamp via a join on `hadm_id`, making them dynamic events.

In [15]:
diag = full[full["code"].str.startswith("DIAGNOSIS")]

print(f"Total diagnoses            : {len(diag):,}")
print(f"With timestamp             : {diag['time'].notna().sum():,}")
print(f"Without timestamp (NaT)    : {diag['time'].isna().sum():,}")
print()
print("Sample diagnosis events:")
diag[["subject_id", "time", "code"]].head(8)

Total diagnoses            : 3,091
With timestamp             : 3,091
Without timestamp (NaT)    : 0

Sample diagnosis events:


,subject_id,time,code
7,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//I251
8,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//I839
9,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//Others
688,7473842122186386,2592-12-14 04:29:00,DIAGNOSIS//KCD8//E149
689,7473842122186386,2592-12-14 04:29:00,DIAGNOSIS//KCD8//Others
1758,14397825242364671,2846-06-20 02:56:00,DIAGNOSIS//KCD8//Others
2939,21501110057397942,2387-11-01 00:00:18,DIAGNOSIS//KCD8//I509
2940,21501110057397942,2387-11-01 00:00:18,DIAGNOSIS//KCD8//Others


---
## 13. Unit Normalization

In [16]:
korean = full[full["code"].str.contains("회|℃|㎍|㎎|×|㎕|μℓ", na=False, regex=True)]
print(f"Codes with non-standard units : {len(korean)}")
print(f"Result: {'✅ All units normalized' if len(korean) == 0 else '❌ Some units not normalized'}")
print()
print("Standard units present in codes (top 10):")
print("-" * 40)
units = full["code"].str.extract(r"//([^/]+)$")[0].dropna().value_counts()
unit_only = units[~units.index.str.match(r"^[0-9A-Z]{3,}_")].head(10)
for unit, count in unit_only.items():
    print(f"  {unit:<15} : {count:,}")

Codes with non-standard units : 0
Result: ✅ All units normalized

Standard units present in codes (top 10):
----------------------------------------
  mmHg            : 187,235
  min             : 67,330
  cc              : 63,044
  %               : 52,771
  cmH2O           : 26,513
  Cel             : 26,467
  ml              : 16,585
  fL              : 7,791
  001L30821       : 4,961
  sec             : 3,996


---
## 14. ICU Procedure Events

In [17]:
proc_start = full[full["code"].str.startswith("PROCEDURE_START")]
proc_end   = full[full["code"].str.startswith("PROCEDURE_END")]

print(f"PROCEDURE_START events     : {len(proc_start):,}")
print(f"PROCEDURE_END events       : {len(proc_end):,}")
print(f"All have timestamps        : {'✅' if proc_start['time'].notna().all() else '❌'}")
print()
print("Most frequent ICU procedures:")
print("-" * 50)
for code, count in proc_start["code"].value_counts().head(5).items():
    print(f"  {code:<40} : {count:,}")

PROCEDURE_START events     : 371,107
PROCEDURE_END events       : 371,107
All have timestamps        : ✅

Most frequent ICU procedures:
--------------------------------------------------
  PROCEDURE_START//001P_OPFG130303         : 33,737
  PROCEDURE_START//001P_OPFG140904         : 33,737
  PROCEDURE_START//001P_L9881              : 33,737
  PROCEDURE_START//001P_RG3011             : 33,737
  PROCEDURE_START//001P_KL1442             : 33,737


---
## 15. Full Validation Summary

In [18]:
dynamic2 = full[full["time"].notna()].copy()
dynamic2["time"] = pd.to_datetime(dynamic2["time"], errors="coerce")
non_proc2 = dynamic2[~dynamic2["code"].str.startswith(("PROCEDURE_START", "PROCEDURE_END"))]

death2 = dynamic2[dynamic2["code"] == "MEDS_DEATH"][["subject_id", "time"]].rename(columns={"time": "death_time"})
latest2 = non_proc2.groupby("subject_id")["time"].max().reset_index().rename(columns={"time": "latest_time"})
death_merged = death2.merge(latest2, on="subject_id") if not death2.empty else pd.DataFrame()
tolerance = pd.Timedelta(hours=48)
death_violations = death_merged[death_merged["death_time"] + tolerance < death_merged["latest_time"]] if not death_merged.empty else pd.DataFrame()

numeric2 = full[full["numeric_value"].notna()].copy()
distrib_ok = all(
    (1 - len(numeric2[numeric2["code"].str.startswith(k) & ((numeric2["numeric_value"] < lo) | (numeric2["numeric_value"] > hi))]) / max(len(numeric2[numeric2["code"].str.startswith(k)]), 1)) * 100 >= 95
    for k, (lo, hi) in CLINICAL_RANGES.items()
    if not numeric2[numeric2["code"].str.startswith(k)].empty
)

print("=" * 55)
print("VALIDATION SUMMARY")
print("=" * 55)
print()

all_checks = {
    # Format
    "All output files present"                      : all((OUTPUT_DIR / f).exists() for f in expected),
    "subject_id is int64"                           : str(train["subject_id"].dtype) in ("int64", "Int64"),
    "time is datetime"                              : "datetime" in str(train["time"].dtype),
    "code is string"                                : str(train["code"].dtype) in ("object", "string", "str"),
    "numeric_value is float32"                      : str(train["numeric_value"].dtype) == "float32",
    # Code quality
    "No //nan in codes"                             : len(train[train["code"].str.contains("//nan", na=False)]) == 0,
    "No UNKNOWN codes"                              : len(train[train["code"] == "UNKNOWN"]) == 0,
    "No Korean units"                               : len(full[full["code"].str.contains("회|℃|㎍|㎎|×|㎕|μℓ", na=False, regex=True)]) == 0,
    # Temporal consistency
    "MEDS_BIRTH is earliest event"                  : True,  # checked above — 0 violations
    "MEDS_DEATH"         : len(death_violations) == 0,
    "ICU_ADMISSION before ICU_DISCHARGE"            : True,  # checked above — 0 violations
    "HOSPITAL_ADMISSION before HOSPITAL_DISCHARGE"  : True,  # checked above — 0 violations
    "No duplicate GENDER events"                    : (full[full["code"].str.startswith("GENDER")].groupby("subject_id").size() > 1).sum() == 0,
    # Distributional
    "Clinical values in plausible ranges (95%)"     : distrib_ok,
    # Cross-cohort
    "Same event types across splits"                : all_same,
    "No patient in multiple splits"                 : overlap == 0,
    "All splits cover >80% of codes"                : all(len(set(df["code"].unique())) / len(all_codes) * 100 >= 80 for df in split_dfs.values()),
    # Metadata
    "Static events are GENDER only"                 : full[full["time"].isna()]["code"].str.startswith("GENDER").all(),
    "Diagnoses have timestamps"                     : full[full["code"].str.startswith("DIAGNOSIS")]["time"].notna().all(),
    "Procedure events present"                      : len(full[full["code"].str.startswith("PROCEDURE_START")]) > 0,
    "Splits 80/10/10"                               : all(split_checks.values()),
    "codes.parquet has descriptions"                : codes["description"].notna().sum() > 0,
    "codes.parquet has parent_codes"                : codes["parent_codes"].notna().sum() > 0,
    "dataset.json is present"                       : (OUTPUT_DIR / "metadata/dataset.json").exists(),
}

for check, result in all_checks.items():
    print(f"  {'✅' if result else '❌'}  {check}")

passed = sum(all_checks.values())
total  = len(all_checks)
print()
print("=" * 55)
print(f"  Result: {passed}/{total} checks passed")
print("=" * 55)

VALIDATION SUMMARY

  ✅  All output files present
  ✅  subject_id is int64
  ✅  time is datetime
  ✅  code is string
  ✅  numeric_value is float32
  ✅  No //nan in codes
  ✅  No UNKNOWN codes
  ✅  No Korean units
  ✅  MEDS_BIRTH is earliest event
  ✅  MEDS_DEATH
  ✅  ICU_ADMISSION before ICU_DISCHARGE
  ✅  HOSPITAL_ADMISSION before HOSPITAL_DISCHARGE
  ✅  No duplicate GENDER events
  ✅  Clinical values in plausible ranges (95%)
  ✅  Same event types across splits
  ✅  No patient in multiple splits
  ✅  All splits cover >80% of codes
  ✅  Static events are GENDER only
  ✅  Diagnoses have timestamps
  ✅  Procedure events present
  ✅  Splits 80/10/10
  ✅  codes.parquet has descriptions
  ✅  codes.parquet has parent_codes
  ✅  dataset.json is present

  Result: 24/24 checks passed


---
## 16. Death Timestamp Precision (`deathtime_map`)

`dod` in K-MIMIC is a date-only field — stored as midnight (`00:00:00`). To improve temporal precision, the pipeline builds a `deathtime_map` from `admissions.deathtime` (which has hour-level precision) and overrides `MEDS_DEATH` timestamps for patients where that field is available.

This section verifies that:
1. The 88 expected patients have a precise (non-midnight) `MEDS_DEATH` timestamp.
2. Their `MEDS_DEATH` time matches the `admissions.deathtime` exactly.
3. The remaining deceased patients (with no `deathtime` in admissions) fall back to `dod` midnight correctly.

In [19]:
INTERMEDIATE_DIR = Path("data/intermediate")

# --- Build deathtime_map from intermediate admissions ---
adm = pd.read_parquet(INTERMEDIATE_DIR / "syn_admissions.parquet")
adm["deathtime"] = pd.to_datetime(adm["deathtime"], errors="coerce")
precise_adm = adm[adm["deathtime"].notna()][["subject_id", "deathtime"]].copy()
deathtime_map = dict(zip(precise_adm["subject_id"], precise_adm["deathtime"]))

# --- Get all MEDS_DEATH events ---
meds_death = full[full["code"] == "MEDS_DEATH"][["subject_id", "time"]].copy()
meds_death["time"] = pd.to_datetime(meds_death["time"], errors="coerce")

# --- Classify each MEDS_DEATH event ---
meds_death["has_precise_deathtime"] = meds_death["subject_id"].isin(deathtime_map)
meds_death["is_midnight"]           = meds_death["time"].dt.time.astype(str) == "00:00:00"
meds_death["expected_time"]         = meds_death["subject_id"].map(deathtime_map)
meds_death["expected_time"]         = pd.to_datetime(meds_death["expected_time"], errors="coerce")
meds_death["matches_deathtime"]     = (
    meds_death["has_precise_deathtime"] &
    (meds_death["time"] == meds_death["expected_time"])
)

total_deaths     = len(meds_death)
precise_patients = meds_death["has_precise_deathtime"].sum()
fallback_patients = total_deaths - precise_patients
precise_non_mid  = (meds_death["has_precise_deathtime"] & ~meds_death["is_midnight"]).sum()
exact_match      = meds_death["matches_deathtime"].sum()

print("Death timestamp precision checks:")
print("-" * 60)
print(f"  Total MEDS_DEATH events         : {total_deaths}")
print(f"  Patients with precise deathtime : {precise_patients}  (from admissions.deathtime)")
print(f"  Patients using dod fallback     : {fallback_patients}")
print()

checks = {
    "88 patients have precise deathtime"    : precise_patients == 88,
    "Precise patients have non-midnight ts" : precise_non_mid == precise_patients,
    "Precise timestamps match admissions"   : exact_match == precise_patients
}
for check, result in checks.items():
    print(f"  {'✅' if result else '❌'}  {check}")

print()
print("Sample — precise deathtime patients (first 5):")
print("-" * 60)
sample = (
    meds_death[meds_death["has_precise_deathtime"]]
    [["subject_id", "time", "expected_time"]]
    .head(5)
    .rename(columns={"time": "MEDS_DEATH_time", "expected_time": "admissions.deathtime"})
)
print(sample.to_string(index=False))

print()
print("Sample — dod fallback patients (first 5):")
print("-" * 60)
fallback_sample = (
    meds_death[~meds_death["has_precise_deathtime"]]
    [["subject_id", "time"]]
    .head(5)
    .rename(columns={"time": "MEDS_DEATH_time (midnight fallback)"})
)
print(fallback_sample.to_string(index=False))

Death timestamp precision checks:
------------------------------------------------------------
  Total MEDS_DEATH events         : 88
  Patients with precise deathtime : 88  (from admissions.deathtime)
  Patients using dod fallback     : 0

  ✅  88 patients have precise deathtime
  ✅  Precise patients have non-midnight ts
  ✅  Precise timestamps match admissions

Sample — precise deathtime patients (first 5):
------------------------------------------------------------
        subject_id     MEDS_DEATH_time admissions.deathtime
 65286727470991881 2551-07-01 09:24:25  2551-07-01 09:24:25
305879959789607859 2136-10-05 15:09:45  2136-10-05 15:09:45
395350095383980000 2449-12-20 07:41:37  2449-12-20 07:41:37
555791147147276508 2505-12-21 07:36:01  2505-12-21 07:36:01
564178785932688137 2078-12-10 03:35:05  2078-12-10 03:35:05

Sample — dod fallback patients (first 5):
------------------------------------------------------------
Empty DataFrame
Columns: [subject_id, MEDS_DEATH_time (midnigh

---
## 17. Patients Without `MEDS_BIRTH`

`MEDS_BIRTH` is generated from `anchor_year - anchor_age` in `syn_patients`. If either field is missing or produces an invalid year, the event is dropped. Out of 1,328 patients, only 1,249 have a `MEDS_BIRTH` event — this section identifies the 79 missing patients and investigates why.

In [20]:
all_patients  = set(full["subject_id"].unique())
birth_patients = set(full[full["code"] == "MEDS_BIRTH"]["subject_id"].unique())
missing_birth  = all_patients - birth_patients

print(f"Total patients              : {len(all_patients):,}")
print(f"Patients with MEDS_BIRTH    : {len(birth_patients):,}")
print(f"Patients without MEDS_BIRTH : {len(missing_birth):,}")
print()

# Check that missing patients still have other events
missing_events = full[full["subject_id"].isin(missing_birth)]
print(f"Missing-MEDS_BIRTH patients still have events  : {missing_events['subject_id'].nunique()} patients")
print(f"Their event count (total)                      : {len(missing_events):,}")
print()

# Investigate source: load intermediate patients to find why
patients_df = pd.read_parquet(INTERMEDIATE_DIR / "syn_patients.parquet")
missing_df  = patients_df[patients_df["subject_id"].isin(missing_birth)][
    ["subject_id", "anchor_age", "anchor_year", "year_of_birth"]
].copy()

print(f"anchor_age missing  : {missing_df['anchor_age'].isna().sum()}")
print(f"anchor_year missing : {missing_df['anchor_year'].isna().sum()}")
print(f"year_of_birth null  : {missing_df['year_of_birth'].isna().sum()}")
print()
print("Sample — patients without MEDS_BIRTH:")
print("-" * 55)
print(missing_df.head(10).to_string(index=False))

Total patients              : 1,328
Patients with MEDS_BIRTH    : 1,249
Patients without MEDS_BIRTH : 79

Missing-MEDS_BIRTH patients still have events  : 79 patients
Their event count (total)                      : 79,165

anchor_age missing  : 79
anchor_year missing : 0
year_of_birth null  : 79

Sample — patients without MEDS_BIRTH:
-------------------------------------------------------
         subject_id  anchor_age  anchor_year  year_of_birth
6956152002738617245         NaN         2999           <NA>
4802739299292145514         NaN         2141           <NA>
6038578796416562984         NaN         2381           <NA>
8712412723170996078         NaN         2768           <NA>
 796721026084912506         NaN         2703           <NA>
8308861362143562191         NaN         2314           <NA>
6867121015015822770         NaN         2160           <NA>
5727965354351462303         NaN         2099           <NA>
5756775694111830178         NaN         2179           <NA>
6565063

---
## 18. Events per Patient Distribution

Verifies that the event load is reasonably distributed across patients — no patient should monopolize the dataset, and no patient should have suspiciously few events.

In [21]:
events_per_patient = full.groupby("subject_id").size()

q = events_per_patient.quantile([0.05, 0.25, 0.5, 0.75, 0.95])

print("Events per patient — distribution:")
print("-" * 40)
print(f"  Min              : {events_per_patient.min():>6,}")
print(f"  5th percentile   : {int(q[0.05]):>6,}")
print(f"  25th percentile  : {int(q[0.25]):>6,}")
print(f"  Median           : {int(q[0.50]):>6,}")
print(f"  Mean             : {events_per_patient.mean():>6,.1f}")
print(f"  75th percentile  : {int(q[0.75]):>6,}")
print(f"  95th percentile  : {int(q[0.95]):>6,}")
print(f"  Max              : {events_per_patient.max():>6,}")
print()

# Outlier thresholds
low_threshold  = 10
high_threshold = events_per_patient.quantile(0.99)

low_outliers  = events_per_patient[events_per_patient < low_threshold]
high_outliers = events_per_patient[events_per_patient > high_threshold]

print(f"Patients with < {low_threshold} events (suspiciously few) : {len(low_outliers)}")
print(f"Patients with > {high_threshold:.0f} events (top 1%)           : {len(high_outliers)}")
print()

checks = {
    "All patients have at least 1 event"    : events_per_patient.min() >= 1,
    "No patient has > 50% of all events"    : events_per_patient.max() / len(full) < 0.5,
    "Median > 100 events per patient"       : int(q[0.5]) > 100,
}
for check, result in checks.items():
    print(f"  {'✅' if result else '❌'}  {check}")

print()
print("Top 5 patients by event count:")
print("-" * 40)
for sid, cnt in events_per_patient.nlargest(5).items():
    pct = cnt / len(full) * 100
    print(f"  subject_id {sid} : {cnt:,} events ({pct:.2f}%)")

print()
print("Bottom 5 patients by event count:")
print("-" * 40)
for sid, cnt in events_per_patient.nsmallest(5).items():
    print(f"  subject_id {sid} : {cnt} events")

Events per patient — distribution:
----------------------------------------
  Min              :    252
  5th percentile   :    660
  25th percentile  :    859
  Median           :  1,057
  Mean             : 1,040.3
  75th percentile  :  1,204
  95th percentile  :  1,437
  Max              :  1,633

Patients with < 10 events (suspiciously few) : 0
Patients with > 1539 events (top 1%)           : 14

  ✅  All patients have at least 1 event
  ✅  No patient has > 50% of all events
  ✅  Median > 100 events per patient

Top 5 patients by event count:
----------------------------------------
  subject_id 4255096729959530848 : 1,633 events (0.12%)
  subject_id 6038578796416562984 : 1,629 events (0.12%)
  subject_id 8066046301780976168 : 1,582 events (0.11%)
  subject_id 8764085030505734917 : 1,579 events (0.11%)
  subject_id 7795075900356440475 : 1,577 events (0.11%)

Bottom 5 patients by event count:
----------------------------------------
  subject_id 1192302099221166119 : 252 events
  su

---
## 19. `INPUT_START` Numeric Value Completeness

`INPUT_START` events represent fluid/drug infusions — they must always carry a numeric value (the administered volume/dose). A missing `numeric_value` here would mean a clinically meaningless event with no quantity recorded.

In [22]:
inputs = full[full["code"].str.startswith("INPUT_START")].copy()

total        = len(inputs)
with_value   = inputs["numeric_value"].notna().sum()
without_value = inputs["numeric_value"].isna().sum()
pct_complete  = with_value / total * 100 if total > 0 else 0

print("INPUT_START numeric value completeness:")
print("-" * 50)
print(f"  Total INPUT_START events     : {total:,}")
print(f"  With numeric_value           : {with_value:,}  ({pct_complete:.1f}%)")
print(f"  Without numeric_value (NaN)  : {without_value:,}")
print()

checks = {
    "INPUT_START events exist"                    : total > 0,
    "All INPUT_START have a numeric value"        : without_value == 0,
    "All numeric values are positive"             : (inputs["numeric_value"] > 0).all(),
}
for check, result in checks.items():
    print(f"  {'✅' if result else '❌'}  {check}")

print()
print("Value distribution (volume/dose):")
print("-" * 50)
q = inputs["numeric_value"].quantile([0.05, 0.25, 0.5, 0.75, 0.95])
print(f"  Min    : {inputs['numeric_value'].min():>10.2f}")
print(f"  P5     : {q[0.05]:>10.2f}")
print(f"  Median : {q[0.50]:>10.2f}")
print(f"  Mean   : {inputs['numeric_value'].mean():>10.2f}")
print(f"  P95    : {q[0.95]:>10.2f}")
print(f"  Max    : {inputs['numeric_value'].max():>10.2f}")
print()
print("Top 5 INPUT_START codes by frequency:")
print("-" * 50)
for code, cnt in inputs["code"].value_counts().head(5).items():
    mean_val = inputs[inputs["code"] == code]["numeric_value"].mean()
    print(f"  {code:<45} : {cnt:,} events  (mean={mean_val:.1f})")

INPUT_START numeric value completeness:
--------------------------------------------------
  Total INPUT_START events     : 26,691
  With numeric_value           : 26,691  (100.0%)
  Without numeric_value (NaN)  : 0

  ✅  INPUT_START events exist
  ✅  All INPUT_START have a numeric value
  ❌  All numeric values are positive

Value distribution (volume/dose):
--------------------------------------------------
  Min    :       0.00
  P5     :      43.75
  Median :     129.44
  Mean   :     138.01
  P95    :     257.36
  Max    :     528.00

Top 5 INPUT_START codes by frequency:
--------------------------------------------------
  INPUT_START//001I_1315_26175//cc              : 21,679 events  (mean=135.7)
  INPUT_START//001I_1301_23155//cc              : 4,221 events  (mean=162.3)
  INPUT_START//001I_1302_21090//cc              : 791 events  (mean=72.0)


---
## 20. Source-to-MEDS Coverage Table

For each of the 15 raw source tables, shows: number of rows in the intermediate Parquet, number of MEDS events produced, and which event types are generated. Tables marked *(metadata only)* are used for code enrichment (descriptions, parent codes) but produce no events directly.

In [23]:
INTERMEDIATE_DIR = Path("data/intermediate")

# Source table → MEDS event type prefixes (from meds_convert.py)
TABLE_EVENT_MAP = {
    "syn_patients":        ["MEDS_BIRTH", "GENDER", "MEDS_DEATH"],
    "syn_admissions":      ["HOSPITAL_ADMISSION", "HOSPITAL_DISCHARGE", "INSURANCE",
                            "MARITAL_STATUS", "ED_REGISTRATION", "ED_OUT"],
    "syn_icustays":        ["ICU_ADMISSION", "ICU_DISCHARGE"],
    "syn_chartevents":     ["CHARTEVENT"],
    "syn_labevents":       ["LAB"],
    "syn_diagnoses_icd":   ["DIAGNOSIS"],
    "syn_procedures_icd":  ["PROCEDURE_ICD"],
    "syn_inputevents":     ["INPUT_START"],
    "syn_outputevents":    ["OUTPUT"],
    "syn_procedureevents": ["PROCEDURE_START", "PROCEDURE_END"],
    "syn_emar":            ["MEDICATION"],
    "syn_d_items":         [],     # metadata enrichment only
    "syn_d_labitems":      [],     # metadata enrichment only
    "syn_emar_detail":     [],     # metadata enrichment only
    "syn_transfers":       [],     # not used in current pipeline
}

_rows = []
for _tname, _prefixes in TABLE_EVENT_MAP.items():
    _ppath = INTERMEDIATE_DIR / f"{_tname}.parquet"
    _src_n = len(pd.read_parquet(_ppath)) if _ppath.exists() else 0
    _meds_n = int(full["code"].str.startswith(tuple(_prefixes)).sum()) if _prefixes else 0
    _rows.append({
        "source_table": _tname,
        "src_rows":     _src_n,
        "meds_events":  _meds_n,
        "event_types":  ", ".join(_prefixes) if _prefixes else "(metadata only)",
    })

_cov = pd.DataFrame(_rows)
_cov["evt/row"] = _cov.apply(
    lambda r: f"{r['meds_events']/r['src_rows']:.2f}" if r["src_rows"] > 0 and r["meds_events"] > 0 else "—",
    axis=1,
)

print(f"Source-to-MEDS coverage — {len(TABLE_EVENT_MAP)} source tables → {_cov['meds_events'].sum():,} MEDS events")
print("=" * 100)
print(f"  {'Source table':<25} {'Src rows':>9} {'MEDS events':>12} {'evt/row':>8}  Event types")
print("-" * 100)
for _, _r in _cov.iterrows():
    print(f"  {_r['source_table']:<25} {_r['src_rows']:>9,} {_r['meds_events']:>12,} {_r['evt/row']:>8}  {_r['event_types']}")
print("-" * 100)
print(f"  {'TOTAL':<25} {_cov['src_rows'].sum():>9,} {_cov['meds_events'].sum():>12,}")
print()

_event_tables = _cov[_cov["event_types"] != "(metadata only)"]
_coverage_checks = {
    "All 15 intermediate tables present"         : all((INTERMEDIATE_DIR / f"{t}.parquet").exists() for t in TABLE_EVENT_MAP),
    "All event-producing tables have MEDS output": (_event_tables["meds_events"] > 0).all(),
}
for _check, _result in _coverage_checks.items():
    print(f"  {'✅' if _result else '❌'}  {_check}")

Source-to-MEDS coverage — 15 source tables → 1,381,580 MEDS events
  Source table               Src rows  MEDS events  evt/row  Event types
----------------------------------------------------------------------------------------------------
  syn_patients                  1,328        2,665     2.01  MEDS_BIRTH, GENDER, MEDS_DEATH
  syn_admissions                1,328        6,449     4.86  HOSPITAL_ADMISSION, HOSPITAL_DISCHARGE, INSURANCE, MARITAL_STATUS, ED_REGISTRATION, ED_OUT
  syn_icustays                  1,328        2,656     2.00  ICU_ADMISSION, ICU_DISCHARGE
  syn_chartevents             416,645      416,645     1.00  CHARTEVENT
  syn_labevents               137,662      137,662     1.00  LAB
  syn_diagnoses_icd             3,091        3,091     1.00  DIAGNOSIS
  syn_procedures_icd            1,479        1,479     1.00  PROCEDURE_ICD
  syn_inputevents              26,691       26,691     1.00  INPUT_START
  syn_outputevents             36,353       36,353     1.00  OUTPUT
 

---
## 21. Code Overlap Analysis

Analyses K-MIMIC codes at three levels of granularity:
- **Exact code** — full code string (e.g. `LAB//001L2001//x10e3/uL`)
- **Event family** — first segment before `//` (e.g. `LAB`, `CHARTEVENT`)
- **EDI parent code** — Korean health-system identifier from `d_labitems` / `d_items`

Set `MIMIC_MEDS_DIR` to a MEDS-formatted MIMIC dataset to run the full K-MIMIC ↔ MIMIC cross-cohort comparison required for the Lane B transfer analysis.

In [27]:
# Code overlap analysis at 3 levels: exact code, event family, parent code
# Set MIMIC_MEDS_DIR to run K-MIMIC ↔ MIMIC cross-cohort comparison
MIMIC_MEDS_DIR = None  # e.g. Path("/path/to/mimic_meds/output")

codes_df = pd.read_parquet(OUTPUT_DIR / "metadata/codes.parquet")

# Level 1: exact code
kmimic_exact = set(codes_df["code"])

# Level 2: event family (first segment before //)
kmimic_families = set(codes_df["code"].str.split("//").str[0])

# Level 3: EDI parent codes
kmimic_parents = set()
for _pc in codes_df["parent_codes"].dropna():
    if isinstance(_pc, list):
        kmimic_parents.update(_pc)
    elif hasattr(_pc, '__iter__') and not isinstance(_pc, str):
        kmimic_parents.update(list(_pc))  # Convert numpy array to list
    else:
        kmimic_parents.add(_pc)

print("K-MIMIC code structure:")
print("-" * 55)
print(f"  Exact codes        : {len(kmimic_exact):>5}  (full code strings)")
print(f"  Event families     : {len(kmimic_families):>5}  (e.g. LAB, CHARTEVENT)")
print(f"  EDI parent codes   : {len(kmimic_parents):>5}  (from d_labitems / d_items)")
print()

print("Codes per event family:")
print("-" * 45)
for _fam, _cnt in codes_df["code"].str.split("//").str[0].value_counts().items():
    print(f"  {_fam:<25} : {_cnt:>4} unique codes")
print()

if MIMIC_MEDS_DIR is not None:
    _mimic_codes = pd.read_parquet(Path(MIMIC_MEDS_DIR) / "metadata/codes.parquet")
    mimic_exact    = set(_mimic_codes["code"])
    mimic_families = set(_mimic_codes["code"].str.split("//").str[0])
    mimic_parents  = set()
    for _pc in _mimic_codes["parent_codes"].dropna():
        mimic_parents.update(_pc if isinstance(_pc, list) else [_pc])

    print("K-MIMIC ↔ MIMIC code overlap:")
    print("-" * 60)
    for _label, (_km, _mi) in {
        "exact code":      (kmimic_exact,    mimic_exact),
        "event family":    (kmimic_families, mimic_families),
        "EDI parent code": (kmimic_parents,  mimic_parents),
    }.items():
        _shared = _km & _mi
        _pct = len(_shared) / len(_km) * 100 if _km else 0
        print(f"  {_label}:")
        print(f"    K-MIMIC  : {len(_km):>5}    MIMIC  : {len(_mi):>5}")
        print(f"    Shared   : {len(_shared):>5}  ({_pct:.1f}% of K-MIMIC codes covered)")
        print(f"    K-only   : {len(_km - _mi):>5}    M-only : {len(_mi - _km):>5}")
        print()
else:
    print("ℹ️  MIMIC_MEDS_DIR not set — cross-cohort overlap analysis skipped.")
    print("   Set MIMIC_MEDS_DIR to a MEDS-formatted MIMIC dataset to enable comparison.")

K-MIMIC code structure:
-------------------------------------------------------
  Exact codes        :   204  (full code strings)
  Event families     :    20  (e.g. LAB, CHARTEVENT)
  EDI parent codes   :    32  (from d_labitems / d_items)

Codes per event family:
---------------------------------------------
  LAB                       :   57 unique codes
  CHARTEVENT                :   23 unique codes
  PROCEDURE_ICD             :   20 unique codes
  DIAGNOSIS                 :   18 unique codes
  MEDICATION                :   13 unique codes
  PROCEDURE_END             :   11 unique codes
  PROCEDURE_START           :   11 unique codes
  ICU_ADMISSION             :   10 unique codes
  ICU_DISCHARGE             :   10 unique codes
  OUTPUT                    :    8 unique codes
  HOSPITAL_ADMISSION        :    4 unique codes
  HOSPITAL_DISCHARGE        :    4 unique codes
  INPUT_START               :    3 unique codes
  INSURANCE                 :    3 unique codes
  MARITAL_STATUS

---
## 22. Tensorisation Timestamp Smoke Test

K-MIMIC uses de-identified years far beyond 2262, which exceeds the `datetime64[ns]` range used by some ML frameworks. This test verifies that timestamps survive a full PyArrow parquet write→read round-trip without overflow, nullification, or precision loss.

Passing this test is a prerequisite for safe use with meds-torch or any downstream tensorisation pipeline.

In [28]:
import tempfile
import pyarrow as pa
import pyarrow.parquet as pq

# Take a representative sample — include extreme de-identified years (> 2262 exceed pandas ns limit)
_dyn = full[full["time"].notna()].copy()
_dyn["time"] = pd.to_datetime(_dyn["time"])
_extreme = _dyn[_dyn["time"].dt.year > 2262]

print("Timestamp range in dataset:")
print("-" * 55)
print(f"  Min timestamp                      : {_dyn['time'].min()}")
print(f"  Max timestamp                      : {_dyn['time'].max()}")
print(f"  Events with year > 2262 (ns-risk)  : {len(_extreme):,}")
print()

# Build test sample: 250 regular + up to 250 extreme-year rows
_test = pd.concat([
    _dyn.head(250),
    _extreme.head(250) if not _extreme.empty else _dyn.tail(250),
], ignore_index=True)[["subject_id", "time", "code", "numeric_value"]]

_schema = pa.schema([
    pa.field("subject_id", pa.int64()),
    pa.field("time",       pa.timestamp("us")),
    pa.field("code",       pa.large_string()),
    pa.field("numeric_value", pa.float32()),
])

with tempfile.TemporaryDirectory() as _tmpdir:
    _path = f"{_tmpdir}/smoke_test.parquet"

    _tbl = pa.Table.from_pandas(_test, schema=_schema, preserve_index=False)
    pq.write_table(_tbl, _path)

    _recovered = pq.read_table(_path).to_pandas()
    _recovered["time"] = pd.to_datetime(_recovered["time"])

    _orig_ts = _test["time"].reset_index(drop=True)
    _recv_ts = _recovered["time"].reset_index(drop=True)

    _ts_match       = (_orig_ts == _recv_ts).all()
    _no_overflow    = _recovered["time"].dt.year.max() == _test["time"].dt.year.max()
    _no_nullified   = _recovered["time"].notna().all()
    _file_kb        = Path(_path).stat().st_size / 1024

print("Tensorisation timestamp smoke test:")
print("-" * 55)
print(f"  Rows tested                : {len(_test):,}")
print(f"  Rows with year > 2262      : {(_test['time'].dt.year > 2262).sum():,}")
print(f"  Written parquet size       : {_file_kb:.1f} KB")
print()

_smoke_checks = {
    "Timestamps survive parquet round-trip"     : bool(_ts_match),
    "No timestamp overflow (year preserved)"    : bool(_no_overflow),
    "No timestamps nullified after write/read"  : bool(_no_nullified),
}
for _check, _result in _smoke_checks.items():
    print(f"  {'✅' if _result else '❌'}  {_check}")

print()
if _ts_match:
    print(f"  datetime64[us] precision fully preserved. Max year in test: {_recovered['time'].dt.year.max()}")
else:
    _n_mismatch = (~(_orig_ts == _recv_ts)).sum()
    print(f"  ❌ {_n_mismatch} timestamp mismatches — investigate before meds-torch use!")

Timestamp range in dataset:
-------------------------------------------------------
  Min timestamp                      : 1949-01-01 00:00:01
  Max timestamp                      : 3021-04-24 18:24:47
  Events with year > 2262 (ns-risk)  : 1,028,387

Tensorisation timestamp smoke test:
-------------------------------------------------------
  Rows tested                : 500
  Rows with year > 2262      : 250
  Written parquet size       : 6.7 KB

  ✅  Timestamps survive parquet round-trip
  ✅  No timestamp overflow (year preserved)
  ✅  No timestamps nullified after write/read

  datetime64[us] precision fully preserved. Max year in test: 2592
